In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

In [0]:
%run ../1_setup/utilities

In [0]:
print(bronze_schema, silver_schema, gold_schema)

In [0]:
dbutils.widgets.text("catalog", "fmcg", "Catalog")
dbutils.widgets.text("data_source", "orders", "Data Source")

catalog = dbutils.widgets.get("catalog")
data_source = dbutils.widgets.get("data_source")

base_path = f"s3://sportsbar-dp-umesh/{data_source}"
landing_path = f"{base_path}/landing/"
processed_path = f"{base_path}/processed/"
print("Base Path: ", base_path)
print("Landing Path: ", landing_path)
print("Processed Path: ", processed_path)

#define the tables
bronze_table = f"{catalog}.{bronze_schema}.{data_source}"
silver_table = f"{catalog}.{silver_schema}.{data_source}"
gold_table = f"{catalog}.{gold_schema}.sb_fact_{data_source}"

In [0]:
df = (spark.read.format("csv")
        .option("header", True)
        .option("inferSchema", True)
        .load(f"{landing_path}/*.csv")
        .withColumn("read_timestamp", F.current_timestamp())
        .select("*", "_metadata.file_name", "_metadata.file_size")
)

print("Total_Rows: ", df.count())
df.show(5)

In [0]:
display(df.limit(20))

In [0]:
df.write\
 .format("delta")\
 .option("delta.enableChangeDataFeed", "true")\
 .mode("append")\
 .saveAsTable(bronze_table)

In [0]:
files = dbutils.fs.ls(landing_path)

for file_info in files:
    dbutils.fs.mv(
        file_info.path, f"{processed_path}/{file_info.name}", True
    )

In [0]:
df_orders = spark.sql(f"select * from {bronze_table}")
df_orders.show(2)

In [0]:
# 1. Keep only rows where order_qty is present.
df_orders = df_orders.filter(F.col("order_qty").isNotNull())

# 2. Clean customer_id, keep numeric else set to 999999
df_orders = df_orders.withColumn("customer_id",
                    F.when(F.col("customer_id").rlike("^[0-9]+$"), F.col("customer_id"))
                     .otherwise("999999")
                     .cast("string")                                 
)

# 3. Remove weekday name from the date text
# "Tuesday, July 01, 2025" -> "July 01, 2025"
df_orders = df_orders.withColumn("order_placement_date",
                    F.regexp_replace(F.col("order_placement_date"), r"^[A-Za-z]+,\s*", "")
)

# 4. Parse order_placement_date using multiple possible formats
df_orders = df_orders.withColumn("order_placement_date", F.coalesce(
                    F.try_to_date("order_placement_date", "yyyy/MM/dd"),
                    F.try_to_date("order_placement_date", "dd-MM-yyyy"),
                    F.try_to_date("order_placement_date", "dd/MM/yyyy"),
                    F.try_to_date("order_placement_date", "MMMM dd, yyyy")
))

# 5. Drop duplicates
df_orders = df_orders.dropDuplicates(["order_id", "order_placement_date", "customer_id", "product_id", "order_qty"])

# 6. convert product_id to string
df_orders = df_orders.withColumn("product_id", F.col("product_id").cast("string"))

In [0]:
# check what's the maximum and minimum date
df_orders.agg(
    F.min("order_placement_date").alias("min_date"),
    F.max("order_placement_date").alias("max_date")
).show()

In [0]:
display(df_orders.limit(20))

In [0]:
df_products = spark.table("fmcg.silver.products")
display(df_products.limit(5))

In [0]:
df_joined = df_orders.join(df_products, on="product_id", how="inner").select(df_orders["*"], df_products["product_code"])

display(df_joined.limit(10))

In [0]:
if not(spark.catalog.tableExists(silver_table)):
    df_joined.write\
        .format("delta")\
        .option("delta.enableChangeDataFeed", "true")\
        .option("mergeSchema", "true")\
        .mode("overwrite")\
        .saveAsTable(silver_table)
else:
    silver_delta = DeltaTable.forName(spark, silver_table)
    silver_delta.alias("silver").merge(
        source = df_joined.alias("bronze"),
        condition = '''
        silver.order_placement_date = bronze.order_placement_date
        AND silver.order_id = bronze.order_id
        AND silver.product_code = bronze.product_code
        AND silver.customer_id = bronze.customer_id
        '''
    ).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()


### Gold

In [0]:
df_gold = spark.sql(f"SELECT order_id, order_placement_date as date, customer_id as customer_code, product_code, product_id, order_qty as sold_quantity FROM {silver_table};")

display(df_gold.limit(2))

In [0]:
if not(spark.catalog.tableExists(gold_table)):
    print("creating new table")
    df_gold.write\
        .format("delta")\
        .option("delta.enableChangeDataFeed", "true")\
        .option("mergeSchema", "true")\
        .mode("overwrite")\
        .saveAsTable(gold_table)
else:
    gold_delta = DeltaTable.forName(spark, gold_table)
    gold_delta.alias("target").merge(
        source = df_gold.alias("source"),
        condition = '''
        target.date = source.date
        AND target.order_id = source.order_id
        AND target.product_code = source.product_code
        AND target.customer_code = source.customer_code
        '''
    ).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

### Merge with Parent Company

In [0]:
df_child = spark.sql(f"SELECT date, product_code, customer_code, sold_quantity FROM {gold_table}")
display(df_child.limit(10))